In [ ]:
!pip install tslearn scikit-learn matplotlib

In [ ]:
import os
from os.path import basename, join
import sys
import joblib

def load_human_action_dataset(data_dir, dataset_name):
    '''
    Loads train and test data from the folder where the
    Human Actions datasets are stored.
    '''
    X_train = joblib.load(os.path.join(data_dir, dataset_name, "X_train.pkl"))
    y_train = joblib.load(os.path.join(data_dir, dataset_name, "y_train.pkl"))
    X_test = joblib.load(os.path.join(data_dir, dataset_name, "X_test.pkl"))
    y_test = joblib.load(os.path.join(data_dir, dataset_name, "y_test.pkl"))

    print("Successfully loaded dataset:", dataset_name)
    print("Size of train data:", len(y_train))
    print("Size of test data:", len(y_test))

    return X_train, y_train, X_test, y_test


In [ ]:
# ctwd_api.py  — RAGGED-FRIENDLY
import numpy as np
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict, Sequence, Union
import heapq

# (tuỳ chọn) dùng SciPy để tăng tốc SpMM; nếu không có vẫn chạy được
try:
    import scipy.sparse as sp
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

BIG = 1e12

# =========================
# 0) HELPERS (ragged / dense)
# =========================
def _as_ragged_list(M: Union[np.ndarray, Sequence[np.ndarray]]) -> Tuple[List[np.ndarray], int]:
    """
    Chuẩn hoá đầu vào về list các mảng (n_i, d).
    Trả về (list_seq, d)
    """
    if isinstance(M, np.ndarray):
        if M.ndim != 3:
            raise ValueError("If ndarray, expect shape (m, n, d).")
        m, n, d = M.shape
        seqs = [M[i] for i in range(m)]
        return seqs, d
    # list/tuple các chuỗi (n_i, d)
    seqs = []
    d = None
    for i, xi in enumerate(M):
        xi = np.asarray(xi, dtype=float)
        if xi.ndim != 2:
            raise ValueError(f"Sequence {i} must have shape (n_i, d).")
        if d is None:
            d = xi.shape[1]
        elif xi.shape[1] != d:
            raise ValueError("All sequences must have the same feature dimension d.")
        seqs.append(xi)
    if d is None:
        raise ValueError("Empty sequence list.")
    return seqs, d

def _linearize_points_ragged(M: Union[np.ndarray, Sequence[np.ndarray]]):
    """
    Hỗ trợ ragged: 
      - P: (N, d) các điểm ghép lại
      - Sidx: (N,) id chuỗi
      - Tpos: (N,) thời gian chuẩn hoá trong [0,1) cho mỗi điểm (i / n_i)
      - lengths: (m_seq,) độ dài từng chuỗi
      - d: số kênh
    """
    seqs, d = _as_ragged_list(M)
    m_seq = len(seqs)
    lengths = np.array([xi.shape[0] for xi in seqs], dtype=int)
    # ghép điểm
    P = np.vstack(seqs) if m_seq > 0 else np.zeros((0, d))
    # id chuỗi
    Sidx = np.repeat(np.arange(m_seq, dtype=int), lengths)
    # vị trí thời gian chuẩn hoá (không dùng endpoint=1 để tránh trùng 1.0)
    Tpos_list = [ (np.arange(n_i, dtype=float) / max(n_i,1)) for n_i in lengths ]
    Tpos = np.concatenate(Tpos_list) if m_seq > 0 else np.zeros((0,), float)
    return P, Sidx, Tpos, m_seq, lengths, d

def _pairwise_sqdist(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """
    Khoảng cách 'lai':
      Euclid^2    các đặc trưng (trừ cột cuối)  +  |orderA - orderB|   (cột cuối)
    """
    assert A.ndim == 2 and B.ndim == 2, "Expect 2D arrays"
    assert A.shape[1] == B.shape[1], "Dim mismatch"
    D = A.shape[1]
    if D == 0:
        return np.zeros((A.shape[0], B.shape[0]), dtype=float)
    if D == 1:
        a_ord = A[:, 0]; b_ord = B[:, 0]
        return np.abs(a_ord[:, None] - b_ord[None, :])
    Af = A[:, :-1]; Bf = B[:, :-1]
    aa = (Af * Af).sum(1)[:, None]
    bb = (Bf * Bf).sum(1)[None, :]
    Dsq = np.clip(aa + bb - 2 * (Af @ Bf.T), 0.0, None)
    a_ord = A[:, -1]; b_ord = B[:, -1]
    Pen = np.abs(a_ord[:, None] - b_ord[None, :])
    return Dsq + Pen

@dataclass
class _Node:
    idx: np.ndarray
    height: float
    left: Optional[int]
    right: Optional[int]
    parent: Optional[int]
    is_leaf: bool

@dataclass
class CTWDModel:
    # shared runtime fields
    P: np.ndarray                  # (N, d_aug) nếu TamLe, hoặc (N, d) nếu Banded
    Sidx: np.ndarray               # (N,) id chuỗi
    Tpos: np.ndarray               # (N,) thời gian chuẩn hoá (0..1)
    lengths: np.ndarray            # (m_seq,)
    m_seq: int
    d: int                         # số kênh gốc (chưa augment)
    nodes: List[_Node]
    leaf_ids: List[int]
    leaf_index_map: Dict[int, int]
    edges: List[Tuple[int, int, float]]     # (parent, child, w_e)
    S_edge_leaf: object                     # (E, L) dense hoặc sp.csr_matrix
    centroids: np.ndarray                   # (num_nodes, D_aug)
    # meta
    mode: str                               # "tamle" | "banded"
    lam_time: float = 0.0
    lam_idx: float = 0.0
    W: float = 0.0                           # band width (tỉ lệ 0..1 cho ragged)
    # caches
    point_leaf: Optional[np.ndarray] = None  # (N,)
    H: Optional[np.ndarray] = None           # (L, m_seq)
    M: Optional[np.ndarray] = None           # (E, m_seq)
    w: Optional[np.ndarray] = None           # (E,)

# =========================
# 1) BOX TREE + GONZALEZ
# =========================
class _KDBoxTree:
    def __init__(self, leaf_size=64, max_depth=24):
        self.leaf_size = leaf_size
        self.max_depth = max_depth
        self.boxes = []
        self.X = None

    def _build(self, idx, depth):
        X = self.X[idx]
        c = X.mean(axis=0)
        r = float(np.sqrt(((X - c) ** 2).sum(1).max())) if X.shape[0] else 0.0
        bid = len(self.boxes)
        self.boxes.append({"idx": idx, "c": c, "r": r, "L": None, "R": None, "leaf": False})
        if idx.size <= self.leaf_size or depth >= self.max_depth or r == 0.0:
            self.boxes[bid]["leaf"] = True
            return bid
        var = X.var(axis=0)
        d = int(np.argmax(var))
        med = np.median(X[:, d])
        mask = X[:, d] <= med
        if mask.all() or (~mask).all():
            mid = idx.size // 2
            Lidx = idx[:mid]; Ridx = idx[mid:]
        else:
            Lidx = idx[mask]; Ridx = idx[~mask]
        L = self._build(Lidx, depth + 1)
        R = self._build(Ridx, depth + 1)
        self.boxes[bid]["L"] = L; self.boxes[bid]["R"] = R
        return bid

    def fit(self, X):
        self.X = X
        self.boxes = []
        self._build(np.arange(X.shape[0]), 0)

def _bounds_box(box, centers):
    if centers.size == 0: return 0.0, float("inf")
    d = np.sqrt(((centers - box["c"][None, :]) ** 2).sum(1))
    dmin = float(d.min())
    r = box["r"]
    return max(0.0, dmin - r), dmin + r

def _farthest_point_by_boxes(X, centers, kdt: _KDBoxTree, gap_tol=1e-6):
    if centers.size == 0: return 0, 0.0
    heap = []
    L0, U0 = _bounds_box(kdt.boxes[0], centers)
    heapq.heappush(heap, (-U0, 0, L0))
    best_idx, best_val = None, -1.0
    while heap:
        negU, bid, Lb = heapq.heappop(heap)
        Ub = -negU
        L2, U2 = _bounds_box(kdt.boxes[bid], centers)
        if U2 < Ub - 1e-12 or L2 > Lb + 1e-12:
            heapq.heappush(heap, (-U2, bid, L2)); continue
        if best_val >= U2 - 1e-15: break
        box = kdt.boxes[bid]
        if box["leaf"] or (U2 - L2) <= gap_tol:
            pts = kdt.X[box["idx"]]
            D = _pairwise_sqdist(pts, centers)  # KHÔNG sqrt
            dmin = D.min(axis=1)
            imax = int(np.argmax(dmin)); val = float(dmin[imax])
            if val > best_val: best_val, best_idx = val, int(box["idx"][imax])
            continue
        for child in (box["L"], box["R"]):
            Lc, Uc = _bounds_box(kdt.boxes[child], centers)
            heapq.heappush(heap, (-Uc, child, Lc))
    return best_idx, best_val

def _gonzalez_box_nlogk(X: np.ndarray, k: int, seed: int,
                        box_leaf_size=64, box_max_depth=24, gap_tol=1e-6):
    rng = np.random.default_rng(seed)
    n = X.shape[0]; assert 1 <= k <= n
    kdt = _KDBoxTree(leaf_size=box_leaf_size, max_depth=box_max_depth); kdt.fit(X)
    i0 = int(rng.integers(0, n)); centers = X[i0:i0+1]; C = [i0]
    for _ in range(1, k):
        idx, _ = _farthest_point_by_boxes(X, centers, kdt, gap_tol)
        C.append(idx); centers = X[np.array(C)]
    return np.array(C, dtype=int)

# =========================
# 1.1) ROUTING & PRECOMPUTE
# =========================
def _route_all_points_vectorized(model: CTWDModel) -> np.ndarray:
    N = model.P.shape[0]
    leaf_of_point = np.empty(N, dtype=np.int32)
    stack = [(0, np.arange(N, dtype=np.int32))]
    nodes = model.nodes; C = model.centroids; P = model.P
    while stack:
        nid, idxs = stack.pop()
        nd = nodes[nid]
        if nd.is_leaf:
            j = model.leaf_index_map[nid]; leaf_of_point[idxs] = j; continue
        L = nd.left; R = nd.right
        X = P[idxs]
        dl = np.linalg.norm(X - C[L], axis=1)
        dr = np.linalg.norm(X - C[R], axis=1)
        go_left = dl <= dr
        if go_left.any():    stack.append((L, idxs[go_left]))
        if (~go_left).any(): stack.append((R, idxs[~go_left]))
    return leaf_of_point

def _precompute_H_M(model: CTWDModel):
    m_seq = model.m_seq
    N = model.P.shape[0]
    L = len(model.leaf_ids)
    E = len(model.edges)
    # 1) route tất cả điểm -> lá
    point_leaf = _route_all_points_vectorized(model)  # (N,)
    model.point_leaf = point_leaf
    # 2) H (L, m_seq) — histogram mỗi chuỗi
    H = np.zeros((L, m_seq), dtype=np.float32)
    for s in range(m_seq):
        mask = (model.Sidx == s)
        if not np.any(mask): continue
        counts = np.bincount(point_leaf[mask], minlength=L).astype(np.float32)
        tot = counts.sum()
        if tot > 0: counts /= tot
        H[:, s] = counts
    model.H = H
    # 3) S_edge_leaf -> CSR (nếu có SciPy) và M = S @ H
    if _HAS_SCIPY:
        SpS = sp.csr_matrix(model.S_edge_leaf)
        model.S_edge_leaf = SpS
        M = (SpS @ H).astype(np.float32)  # (E, m_seq)
    else:
        M = (model.S_edge_leaf @ H).astype(np.float32)
    model.M = M
    # 4) Trọng số cạnh
    model.w = np.array([we for _, _, we in model.edges], dtype=np.float32)

# =========================
# 1.2) CTWD — TAM LE (ragged OK)
# =========================
def _augment_points(seq: np.ndarray, lam_time: float) -> np.ndarray:
    n = seq.shape[0]
    t = (np.arange(n, dtype=float) / max(n, 1))[:, None] * np.sqrt(lam_time)
    return np.hstack([seq, t])

def build_ctwd_tamle(
    M: Union[np.ndarray, Sequence[np.ndarray]],
    lam_time: float = 5.0,
    leaf_size: int = 16,
    max_depth: int = 20,
    seed: int = 0,
    k_split: int = 2,
    box_leaf_size: int = 64,
    box_max_depth: int = 24,
) -> CTWDModel:
    """
    Xây 1 cây global theo TamLe (augment theo thời gian chuẩn hoá → ragged friendly).
    """
    P_raw, Sidx, Tpos, m_seq, lengths, d = _linearize_points_ragged(M)
    # augment từng chuỗi rồi ghép
    P_aug_list = []
    start = 0
    for s in range(m_seq):
        n_i = lengths[s]
        seq = P_raw[start:start+n_i]
        P_aug_list.append(_augment_points(seq, lam_time))
        start += n_i
    P_aug = np.vstack(P_aug_list) if P_aug_list else np.zeros((0, d+1))
    # build tree
    nodes: List[_Node] = []; leaf_ids: List[int] = []
    def _euclid_radius(X):
        if X.shape[0] <= 1: return 0.0
        if X.shape[0] > 1024:
            I = np.random.default_rng(0).choice(X.shape[0], 1024, replace=False); Y = X[I]
        else: Y = X
        j0 = 0; d0 = np.linalg.norm(Y - Y[j0], axis=1); j1 = int(np.argmax(d0))
        d1 = np.linalg.norm(Y - Y[j1], axis=1); return 0.5 * float(d1.max())
    def build(idx: np.ndarray, depth: int, parent: Optional[int], seed_: int) -> int:
        Xsub = P_aug[idx]; h = _euclid_radius(Xsub)
        nid = len(nodes); nodes.append(_Node(idx, h, None, None, parent, False))
        if idx.size <= leaf_size or depth >= max_depth or h == 0.0:
            nodes[nid].is_leaf = True; leaf_ids.append(nid); return nid
        C = _gonzalez_box_nlogk(Xsub, k=k_split, seed=seed_,
                                box_leaf_size=box_leaf_size, box_max_depth=box_max_depth)
        centers = Xsub[C]
        lab = np.argmin(_pairwise_sqdist(Xsub, centers), axis=1)
        if k_split == 2:
            left_idx = idx[lab == 0]; right_idx = idx[lab != 0]
        else:
            cnt = np.bincount(lab, minlength=k_split); main = int(np.argmax(cnt))
            left_idx = idx[lab == main]; right_idx = idx[lab != main]
        if left_idx.size == 0 or right_idx.size == 0:
            mid = idx.size // 2; left_idx = idx[:mid]; right_idx = idx[mid:]
        L = build(left_idx, depth+1, nid, seed_+1); R = build(right_idx, depth+1, nid, seed_+2)
        nodes[nid].left, nodes[nid].right = L, R; return nid
    _ = build(np.arange(P_aug.shape[0]), 0, None, seed)
    # edges & structures
    edges = []
    for cid, nd in enumerate(nodes):
        if nd.parent is not None:
            p = nodes[nd.parent]; w = max(0.0, p.height - nd.height)
            edges.append((nd.parent, cid, w))
    leaf_index_map = {nid: i for i, nid in enumerate(leaf_ids)}
    E, L = len(edges), len(leaf_ids)
    S_edge_leaf = np.zeros((E, L), dtype=np.float32)
    def collect_leaves(nid, out):
        nd = nodes[nid]
        if nd.is_leaf: out.append(nid); return
        if nd.left is not None: collect_leaves(nd.left, out)
        if nd.right is not None: collect_leaves(nd.right, out)
    for e, (pid, cid, _) in enumerate(edges):
        leaves = []; collect_leaves(cid, leaves)
        for ln in leaves:
            j = leaf_index_map[ln]; S_edge_leaf[e, j] = 1.0
    centroids = np.vstack([P_aug[nd.idx].mean(axis=0) for nd in nodes])
    model = CTWDModel(
        P=P_aug, Sidx=Sidx, Tpos=Tpos, lengths=lengths, m_seq=m_seq, d=d,
        nodes=nodes, leaf_ids=leaf_ids, leaf_index_map=leaf_index_map,
        edges=edges, S_edge_leaf=S_edge_leaf, centroids=centroids,
        mode="tamle", lam_time=lam_time
    )
    _precompute_H_M(model)
    return model

# =========================
# 2) CTWD — BANDED (ragged OK, W là TỈ LỆ 0..1)
# =========================
def build_ctwd_banded(
    M: Union[np.ndarray, Sequence[np.ndarray]],
    W: Union[int, float] = 0.15,   # nếu <=1 → coi là tỉ lệ; nếu >1 → sẽ chuyển sang tỉ lệ theo n_i
    lam_idx: float = 5.0,
    lam_tree: float = 10.0,
    leaf_size: int = 32,
    max_depth: int = 20,
    seed: int = 0,
) -> CTWDModel:
    # Linearize (không augment)
    P, Sidx, Tpos, m_seq, lengths, d = _linearize_points_ragged(M)
    # Chuyển W về tỉ lệ (0..1)
    if isinstance(W, (int, float)):
        if W <= 1.0:
            W_frac = float(W)
        else:
            # với ragged: xấp xỉ bằng trung bình W/n_i
            W_frac = float(W) / max(1, int(np.median(lengths)))
    else:
        W_frac = 0.15

    nodes: List[_Node] = []; leaf_ids: List[int] = []

    def _euclid_radius(X):
        if X.shape[0] <= 1: return 0.0
        if X.shape[0] > 1024:
            I = np.random.default_rng(0).choice(X.shape[0], 1024, replace=False); Y = X[I]
        else: Y = X
        j0 = 0; d0 = np.linalg.norm(Y - Y[j0], axis=1); j1 = int(np.argmax(d0))
        d1 = np.linalg.norm(Y - Y[j1], axis=1); return 0.5 * float(d1.max())

    def _gonz2_banded(idx_local, seed_):
        Xsub = P[idx_local]; Tsub = Tpos[idx_local]
        rng = np.random.default_rng(seed_)
        i0 = int(rng.integers(0, Xsub.shape[0]))
        # farthest từ i0 với phạt theo thời gian chuẩn hoá
        eu = np.linalg.norm(Xsub - Xsub[i0], axis=1)
        dt = np.abs(Tsub - Tsub[i0])
        c = eu + lam_idx * (dt ** 2)
        c[dt > W_frac] = BIG
        i1 = int(np.argmax(c))
        # farthest từ i1
        eu = np.linalg.norm(Xsub - Xsub[i1], axis=1)
        dt = np.abs(Tsub - Tsub[i1])
        c = eu + lam_idx * (dt ** 2)
        c[dt > W_frac] = BIG
        i2 = int(np.argmax(c))
        return i1, i2

    def build(idx: np.ndarray, depth: int, parent: Optional[int], seed_: int) -> int:
        Xsub = P[idx]; h = lam_tree * _euclid_radius(Xsub)
        nid = len(nodes); nodes.append(_Node(idx, h, None, None, parent, False))
        if idx.size <= leaf_size or depth >= max_depth or h == 0.0:
            nodes[nid].is_leaf = True; leaf_ids.append(nid); return nid
        i1_local, i2_local = _gonz2_banded(idx, seed_)
        c1, t1 = P[idx[i1_local]], Tpos[idx[i1_local]]
        c2, t2 = P[idx[i2_local]], Tpos[idx[i2_local]]
        eu1 = np.linalg.norm(P[idx] - c1, axis=1); eu2 = np.linalg.norm(P[idx] - c2, axis=1)
        dt1 = np.abs(Tpos[idx] - t1); dt2 = np.abs(Tpos[idx] - t2)
        cost1 = eu1 + lam_idx * (dt1 ** 2); cost2 = eu2 + lam_idx * (dt2 ** 2)
        cost1[dt1 > W_frac] = BIG; cost2[dt2 > W_frac] = BIG
        both_big = (cost1 >= BIG) & (cost2 >= BIG)
        # fallback nếu cả hai ngoài băng: bỏ ràng buộc
        cost1[both_big] = eu1[both_big]; cost2[both_big] = eu2[both_big]
        left_mask = cost1 <= cost2
        if left_mask.all() or (~left_mask).all():
            mid = idx.size // 2; left_idx = idx[:mid]; right_idx = idx[mid:]
        else:
            left_idx = idx[left_mask]; right_idx = idx[~left_mask]
        L = build(left_idx, depth+1, nid, seed_+1); R = build(right_idx, depth+1, nid, seed_+2)
        nodes[nid].left, nodes[nid].right = L, R; return nid

    _ = build(np.arange(P.shape[0]), 0, None, seed)

    edges = []
    for cid, nd in enumerate(nodes):
        if nd.parent is not None:
            p = nodes[nd.parent]; w = max(0.0, p.height - nd.height)
            edges.append((nd.parent, cid, w))

    leaf_index_map = {nid: i for i, nid in enumerate(leaf_ids)}
    E, L = len(edges), len(leaf_ids)
    S_edge_leaf = np.zeros((E, L), dtype=np.float32)
    def collect_leaves(nid, out):
        nd = nodes[nid]
        if nd.is_leaf: out.append(nid); return
        if nd.left is not None: collect_leaves(nd.left, out)
        if nd.right is not None: collect_leaves(nd.right, out)
    for e, (pid, cid, _) in enumerate(edges):
        leaves = []; collect_leaves(cid, leaves)
        for ln in leaves:
            j = leaf_index_map[ln]; S_edge_leaf[e, j] = 1.0

    centroids = np.vstack([P[nd.idx].mean(axis=0) for nd in nodes])

    model = CTWDModel(
        P=P, Sidx=Sidx, Tpos=Tpos, lengths=lengths, m_seq=m_seq, d=d,
        nodes=nodes, leaf_ids=leaf_ids, leaf_index_map=leaf_index_map,
        edges=edges, S_edge_leaf=S_edge_leaf, centroids=centroids,
        mode="banded", lam_idx=lam_idx, W=float(W_frac)
    )
    _precompute_H_M(model)
    return model

# =========================
# 3) DISTANCE APIs
# =========================
def ctwd_between_series_fast(model: CTWDModel, s_ref: int, s_cmp: int) -> float:
    """
    CTWD(s_ref, s_cmp) với cache:
      cost = sum_e w_e * |M[e, s_ref] - M[e, s_cmp]|
    """
    w = model.w; M = model.M
    diff = np.abs(M[:, s_ref] - M[:, s_cmp])
    return float((w * diff).sum())

def ctwd_between_series(model: CTWDModel, s_ref: int, s_cmp: int, p: int = 1) -> float:
    assert p == 1, "Hiện tại hỗ trợ p=1 (W1    cây)."
    return ctwd_between_series_fast(model, s_ref, s_cmp)


In [ ]:
# run_tsne_all_ctwd_dtw.py
import numpy as np
import time
import os
import pandas as pd

from tslearn.metrics import dtw
from tslearn.datasets import UCR_UEA_datasets

from sklearn.manifold import TSNE, MDS
from sklearn.metrics import pairwise_distances

# ---------- utils ----------
def as_ragged_list(X):
    if isinstance(X, np.ndarray) and X.ndim == 3 and X.dtype != object:
        return [X[i] for i in range(X.shape[0])]
    return [np.asarray(x, dtype=float) for x in X]


def zscore_per_series(X_list, eps: float = 1e-8):
    """Chuẩn hoá z-score từng chuỗi riêng lẻ (shape-based)."""
    out = []
    for x in X_list:
        x = np.asarray(x, dtype=float)
        mu = x.mean(axis=0, keepdims=True)
        sigma = x.std(axis=0, keepdims=True)
        sigma[sigma < eps] = 1.0
        out.append((x - mu) / sigma)
    return out


def build_DTW_matrix(X_list, sakoe=None):
    n = len(X_list)
    D = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(i + 1, n):
            d = dtw(
                X_list[i],
                X_list[j],
                global_constraint="sakoe_chiba" if sakoe is not None else None,
                sakoe_chiba_radius=sakoe if sakoe is not None else None,
            )
            D[i, j] = D[j, i] = float(d)
    return D


# ---------- CTWD ----------
def build_CTWD_matrix(
    X_list,
    lam_time=0.5,
    leaf_size=4,
    max_depth=8,
    seed=0,
):
    """
    Xây dựng ma trận khoảng cách CTWD trên chuỗi đã chuẩn hoá per-series.
    Không dùng thêm global z-score để giữ đúng "shape" đã chuẩn hoá.
    Cần có các hàm:
      - build_ctwd_tamle
      - ctwd_between_series_fast
    trong namespace.
    """
    X_norm = X_list  # đã z-score per-series từ ngoài

    start_time_tree = time.time()
    model = build_ctwd_tamle(
        X_norm,
        lam_time=lam_time,
        leaf_size=leaf_size,
        max_depth=max_depth,
        seed=seed,
    )
    time_tree = time.time() - start_time_tree

    n = len(X_norm)
    D = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(i + 1, n):
            dij = ctwd_between_series_fast(model, i, j)
            D[i, j] = D[j, i] = dij

    return D, time_tree


# ---------- t-SNE helpers ----------
def suggest_perplexity(m: int) -> int:
    """Giống code t-SNE trước: sqrt(N) nhưng trong [5, 40] và không vượt quá (N-1)//3-1."""
    base = int(np.clip(int(np.sqrt(max(1, m))), 5, 40))
    max_ok = max(5, (m - 1) // 3 - 1)
    return max(5, min(base, max_ok))


def tsne_from_distance(D, random_state: int = 42, dataset_name: str = "", method_name: str = ""):
    """Chạy MDS (non-metric) để lấy init rồi t-SNE với metric='precomputed'."""
    m = D.shape[0]

    # MDS init
    mds = MDS(
        n_components=2,
        dissimilarity="precomputed",
        metric=False,
        random_state=random_state,
        n_init=4,
        max_iter=800,
        eps=1e-3,
    )
    init_2d = mds.fit_transform(D)

    # t-SNE
    perplexity = suggest_perplexity(m)
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        learning_rate=200,
        metric="precomputed",
        method="exact",
        init=init_2d,
        random_state=random_state,
        verbose=1,
    )
    Y = tsne.fit_transform(D)
    return Y, perplexity


# ---------- Quality metrics: Trustworthiness, Continuity, LCMC ----------
def _rank_matrix_from_distance(D: np.ndarray) -> np.ndarray:
    """
    Tạo ma trận rank r_X(i, j) từ ma trận khoảng cách D (0 trên đường chéo).
    r_X(i, j) = 1,2,...,n-1 (không tính self).
    """
    n = D.shape[0]
    ranks = np.zeros_like(D, dtype=int)
    for i in range(n):
        order = np.argsort(D[i])  # từ nhỏ đến lớn
        rank = 1
        for idx in order:
            if idx == i:
                continue
            ranks[i, idx] = rank
            rank += 1
    return ranks


def trustworthiness_from_distance(D_high: np.ndarray, Y: np.ndarray, k: int) -> float:
    """
    Trustworthiness T(k) sử dụng khoảng cách D_high cho không gian gốc
    và khoảng cách Euclidean trong Y cho không gian nhúng.
    """
    n = D_high.shape[0]
    Dh = D_high
    Dl = pairwise_distances(Y, metric="euclidean")

    ranks_high = _rank_matrix_from_distance(Dh)

    T_sum = 0.0
    for i in range(n):
        # k-láng giềng gần nhất trong embedding (trừ self)
        neigh_low = np.argsort(Dl[i])[1 : k + 1]
        neigh_high_set = set(np.argsort(Dh[i])[1 : k + 1])

        for j in neigh_low:
            if j not in neigh_high_set:
                T_sum += ranks_high[i, j] - k

    normalizer = 2.0 / (n * k * (2 * n - 3 * k - 1))
    return 1.0 - normalizer * T_sum


def continuity_from_distance(D_high: np.ndarray, Y: np.ndarray, k: int) -> float:
    """
    Continuity C(k): đối xứng của Trustworthiness, phạt điểm nếu
    láng giềng trong không gian gốc không còn là láng giềng trong embedding.
    """
    n = D_high.shape[0]
    Dh = D_high
    Dl = pairwise_distances(Y, metric="euclidean")

    ranks_low = _rank_matrix_from_distance(Dl)

    C_sum = 0.0
    for i in range(n):
        neigh_high = np.argsort(Dh[i])[1 : k + 1]
        neigh_low_set = set(np.argsort(Dl[i])[1 : k + 1])

        for j in neigh_high:
            if j not in neigh_low_set:
                C_sum += ranks_low[i, j] - k

    normalizer = 2.0 / (n * k * (2 * n - 3 * k - 1))
    return 1.0 - normalizer * C_sum


def lcmc_from_distance(D_high: np.ndarray, Y: np.ndarray, k: int) -> float:
    """
    LCMC(k) = Q_NXNY(k) - k/(n-1),
    với Q_NXNY(k) = (1/(n*k)) * Σ_i |N_X(i,k) ∩ N_Y(i,k)|.
    """
    n = D_high.shape[0]
    Dh = D_high
    Dl = pairwise_distances(Y, metric="euclidean")

    total_overlap = 0.0
    for i in range(n):
        neigh_h = set(np.argsort(Dh[i])[1 : k + 1])
        neigh_l = set(np.argsort(Dl[i])[1 : k + 1])
        total_overlap += len(neigh_h & neigh_l)

    Q = total_overlap / (n * k)
    Q_rand = k / (n - 1)
    return Q - Q_rand


# ---------- Plot helper ----------
import matplotlib.pyplot as plt


def plot_tsne(
    Y: np.ndarray,
    y_true: np.ndarray,
    dataset_name: str,
    method_name: str,
    out_dir: str,
):
    """
    Vẽ scatter t-SNE, phân biệt class bằng màu, lưu PNG.
    """
    os.makedirs(out_dir, exist_ok=True)

    classes = np.unique(y_true)
    cmap = plt.cm.get_cmap("tab10", len(classes))

    plt.figure(figsize=(7.6, 5))
    for k_idx, cls in enumerate(classes):
        mask = (y_true == cls)
        plt.scatter(
            Y[mask, 0],
            Y[mask, 1],
            s=28,
            marker="o",
            alpha=0.85,
            color=cmap(k_idx),
            label=str(cls),
        )

    plt.title(f"t-SNE ({method_name}) — {dataset_name}")
    plt.legend(fontsize=9, ncol=2, frameon=False)
    plt.tight_layout()

    fname = f"{dataset_name}_TSNE_{method_name}.png"
    out_path = os.path.join(out_dir, fname)
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[INFO] Saved plot: {out_path}")


# ---------- main runner ----------
def run_tsne_compare(
    dataset_name,
    Xtr,
    ytr,
    Xte,
    yte,
    sakoe=5,
    lam_time=0.5,
    leaf_size=4,
    max_depth=8,
    base_seed=0,
    n_runs=5,
    k_quality=5,
    plot_dir="tsne_plots",
):
    # 1) Thống kê dataset
    n_train = len(Xtr)
    n_test = len(Xte)

    if isinstance(Xtr, np.ndarray) and Xtr.ndim == 3:
        n_features = Xtr.shape[2]
    elif hasattr(Xtr[0], "ndim") and Xtr[0].ndim > 1:
        n_features = Xtr[0].shape[1]
    else:
        n_features = 1

    y = np.concatenate([np.asarray(ytr), np.asarray(yte)])
    _, y_true = np.unique(y, return_inverse=True)  # encode 0..C-1

    n_classes = len(np.unique(y_true))
    m = len(y_true)

    print(f"\n=== Dataset {dataset_name} ===")
    print(f"| Train: {n_train} | Test: {n_test} | Tổng: {n_train + n_test}")
    print(f"| Số chiều (Features): {n_features} | Số nhãn (Classes): {n_classes}")
    print(f"| lam_time (CTWD): {lam_time}, leaf_size={leaf_size}, max_depth={max_depth}")
    print(f"| n_runs (t-SNE): {n_runs}, k_quality={k_quality}")
    print("=====================================")

    # 2) Chuẩn bị dữ liệu gốc (ragged)
    Xtr_list_raw = as_ragged_list(Xtr)
    Xte_list_raw = as_ragged_list(Xte)

    # 3) Lists thống kê cho nhiều run
    ctwd_T_list, ctwd_C_list, ctwd_L_list = [], [], []
    dtw_T_list, dtw_C_list, dtw_L_list = [], [], []

    # 4) Lặp n_runs: mỗi run build lại CTWD tree + ma trận, DTW matrix, t-SNE
    for run_idx in range(n_runs):
        run_seed = base_seed + run_idx
        print(f"\n  [Run {run_idx + 1}/{n_runs}] seed={run_seed}")

        # 4.1) Chuẩn hoá per-series (mỗi run có thể giống nhau, nhưng giữ nguyên logic cũ)
        Xtr_list = zscore_per_series(Xtr_list_raw)
        Xte_list = zscore_per_series(Xte_list_raw)
        X_list = Xtr_list + Xte_list  # toàn bộ m chuỗi

        # 4.2) CTWD distance
        print("    [CTWD] Building distance matrix ...")
        D_ctwd, time_ctwd_tree = build_CTWD_matrix(
            X_list,
            lam_time=lam_time,
            leaf_size=leaf_size,
            max_depth=max_depth,
            seed=run_seed,
        )
        print(f"    [CTWD] Done. Tree build time = {time_ctwd_tree:.4f}s")
        

        # 4.4) t-SNE với CTWD
        print("    [t-SNE CTWD] Running t-SNE ...")
        Y_ctwd, perp_ctwd = tsne_from_distance(D_ctwd, random_state=run_seed,
                                               dataset_name=dataset_name, method_name="CTWD")
        T_ctwd = trustworthiness_from_distance(D_ctwd, Y_ctwd, k_quality)
        C_ctwd = continuity_from_distance(D_ctwd, Y_ctwd, k_quality)
        L_ctwd = lcmc_from_distance(D_ctwd, Y_ctwd, k_quality)
        ctwd_T_list.append(T_ctwd)
        ctwd_C_list.append(C_ctwd)
        ctwd_L_list.append(L_ctwd)
        print(f"    [CTWD] Perplexity={perp_ctwd}, T={T_ctwd:.4f}, C={C_ctwd:.4f}, LCMC={L_ctwd:.4f}")

        # Chỉ lưu plot của run đầu tiên (tránh ghi đè nhiều file tương tự)
        if run_idx == 0:
            plot_tsne(
                Y_ctwd,
                y_true,
                dataset_name=dataset_name,
                method_name="CTWD",
                out_dir=plot_dir,
            )
    
        if run_idx == 0:
        # 4.3) DTW distance
            print("    [DTW] Building distance matrix ...")
            D_dtw = build_DTW_matrix(X_list, sakoe=sakoe)
            print("    [DTW] Done.")
            # 4.5) t-SNE với DTW
            print("    [t-SNE DTW] Running t-SNE ...")
            Y_dtw, perp_dtw = tsne_from_distance(D_dtw, random_state=run_seed,
                                                 dataset_name=dataset_name, method_name="DTW")
            T_dtw = trustworthiness_from_distance(D_dtw, Y_dtw, k_quality)
            C_dtw = continuity_from_distance(D_dtw, Y_dtw, k_quality)
            L_dtw = lcmc_from_distance(D_dtw, Y_dtw, k_quality)
            dtw_T_list.append(T_dtw)
            dtw_C_list.append(C_dtw)
            dtw_L_list.append(L_dtw)
            print(f"    [DTW ] Perplexity={perp_dtw}, T={T_dtw:.4f}, C={C_dtw:.4f}, LCMC={L_dtw:.4f}")

        if run_idx == 0:
            plot_tsne(
                Y_dtw,
                y_true,
                dataset_name=dataset_name,
                method_name="DTW",
                out_dir=plot_dir,
            )

    # 5) Gom thống kê (mean/var theo n_runs)
    results = {
        "Dataset": dataset_name,
        "Num_Samples": m,

        # CTWD metrics
        "CTWD_Trust_Mean": np.mean(ctwd_T_list),
        "CTWD_Trust_Var": np.var(ctwd_T_list),
        "CTWD_Continuity_Mean": np.mean(ctwd_C_list),
        "CTWD_Continuity_Var": np.var(ctwd_C_list),
        "CTWD_LCMC_Mean": np.mean(ctwd_L_list),
        "CTWD_LCMC_Var": np.var(ctwd_L_list),

        # DTW metrics
        "DTW_Trust_Mean": np.mean(dtw_T_list),
        "DTW_Trust_Var": np.var(dtw_T_list),
        "DTW_Continuity_Mean": np.mean(dtw_C_list),
        "DTW_Continuity_Var": np.var(dtw_C_list),
        "DTW_LCMC_Mean": np.mean(dtw_L_list),
        "DTW_LCMC_Var": np.var(dtw_L_list),
    }

    return results


# ---------- main ----------
if __name__ == "__main__":
    ucruea = UCR_UEA_datasets()

    output_filename = "tsne_quality_all_ctwd_vs_dtw_k5.csv"
    plot_dir = "tsne_plots_ctwd_dtw"
    all_results = []

    # 20 UCR datasets bạn đang dùng
    ucr_uea_datasets = [
        "ArrowHead",              # AH
        "BasicMotions",           # BM
        "BeetleFly",              # BF
        "CBF",                    # CBF
        "Chinatown",              # CT
        "CinCECGTorso",           # CET
        "DiatomSizeReduction",    # DSR
        "GunPointAgeSpan",        # GPA
        "GunPointMaleVersusFemale", # GPM
        "GunPointOldVersusYoung", # GPO
        "Ham",                    # Ham
        "InsectEPGRegularTrain",  # IERT
        "ItalyPowerDemand",       # IPD
        "Meat",                   # Meat
        "MelbournePedestrian",    # MP
        "MixedShapesSmallTrain",  # MS2T
        "MoteStrain",             # MS
        "OliveOil",               # O2
        "Plane",                  # Plane
        "SmoothSubspace",         # S2
    ]

    # Nhóm dataset phi tuyến / local-pattern: lam_time nhỏ
    nonlinear_datasets = {
        "ArrowHead",
        "BeetleFly",
        "CBF",
        "CinCECGTorso",
        "DiatomSizeReduction",
        "Meat",
        "MoteStrain",
        "OliveOil",
        "MixedShapesSmallTrain",
        "Plane",
        "SmoothSubspace",
        "InsectEPGRegularTrain",
    }

    # Nhóm tuyến tính / yếu cấu trúc: lam_time lớn
    linear_datasets = {
        "Chinatown",
        "MelbournePedestrian",
        "Ham",
        "ItalyPowerDemand",
        "GunPointAgeSpan",
        "GunPointMaleVersusFemale",
        "GunPointOldVersusYoung",
    }

    lam_time_small = 0.5   # phi tuyến: giảm penalty thời gian (gần DTW hơn)
    lam_time_mid   = 10.0   # mặc định
    lam_time_large = 30.0  # tuyến tính: nhấn mạnh cấu trúc thời gian

    N_RUNS = 5             # số run t-SNE (có thể tăng nếu muốn lấy trung bình)
    SAKOE  = 5             # Sakoe-Chiba radius cho DTW
    LEAF_SIZE = 4
    MAX_DEPTH = 8
    BASE_SEED = 0
    K_QUALITY = 5          # k cho Trustworthiness, Continuity, LCMC

    print(f"--- Running t-SNE (CTWD vs DTW) for {len(ucr_uea_datasets)} UCR datasets ---")
    print(f"n_runs per dataset = {N_RUNS}, k_quality = {K_QUALITY}")

    for dataset_name in ucr_uea_datasets:
        try:
            Xtr, ytr, Xte, yte = ucruea.load_dataset(dataset_name)

            if dataset_name in nonlinear_datasets:
                lam_time = lam_time_small
            elif dataset_name in linear_datasets:
                lam_time = lam_time_large
            else:
                lam_time = lam_time_mid

            result = run_tsne_compare(
                dataset_name,
                Xtr,
                ytr,
                Xte,
                yte,
                sakoe=SAKOE,
                lam_time=lam_time,
                leaf_size=LEAF_SIZE,
                max_depth=MAX_DEPTH,
                base_seed=BASE_SEED,
                n_runs=N_RUNS,
                k_quality=K_QUALITY,
                plot_dir=plot_dir,
            )

            all_results.append(result)
            df = pd.DataFrame(all_results)
            print(df)

            df.to_csv(
                output_filename,
                index=False,
                float_format="%.6f",
            )
            print(f"✅ Đã lưu kết quả của {dataset_name} vào {output_filename}")

        except Exception as e:
            print(f"!! ERROR processing {dataset_name}: {e}")
            print("   Skipping this dataset.")

    if all_results:
        print(f"\n✅ Hoàn thành. Đã lưu {len(all_results)} dòng vào: {output_filename}")
        print(f"Ảnh t-SNE được lưu trong thư mục: {plot_dir}")
    else:
        print("\n⚠️ Không có dataset nào chạy thành công.")
